In [32]:
import sys
print(sys.executable)
print(sys.version)

/opt/homebrew/opt/python@3.10/bin/python3.10
3.10.18 (main, Jun  3 2025, 18:23:41) [Clang 17.0.0 (clang-1700.0.13.3)]


In [6]:
!python3 -m venv venv
!source venv/bin/activate

In [13]:
!pip3 install ultralytics
!pip3 install opencv-python
!pip3 install torch
!pip3 install matplotlib
!pip3 install fastapi uvicorn

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3

<h1> Dataset preprocessing </h1>

In [40]:
import os
import cv2
import numpy as np
import shutil
from sklearn.model_selection import train_test_split

# Paths
MVTEC_ROOT = "dataset/mvtec_anomaly_detection"
OUTPUT_ROOT = "dataset_yolo"

IMG_TRAIN = os.path.join(OUTPUT_ROOT, "images/train")
IMG_VAL = os.path.join(OUTPUT_ROOT, "images/val")
LBL_TRAIN = os.path.join(OUTPUT_ROOT, "labels/train")
LBL_VAL = os.path.join(OUTPUT_ROOT, "labels/val")

for p in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
    os.makedirs(p, exist_ok=True)

# Defect categories
MINOR_DEFECTS = ["scratch", "contamination", "color", "thread_side", "glue"]
MAJOR_DEFECTS = ["crack", "broken", "hole", "cut", "faulty"]

# Class mapping function
def get_class(defect):
    if defect in MINOR_DEFECTS:
        return 0
    elif defect in MAJOR_DEFECTS:
        return 1
    else:
        return 2  # good

image_paths = []
labels = []

print("Scanning MVTec dataset...")

for category in os.listdir(MVTEC_ROOT):
    category_path = os.path.join(MVTEC_ROOT, category)
    if not os.path.isdir(category_path):
        continue

    test_path = os.path.join(category_path, "test")
    gt_path = os.path.join(category_path, "ground_truth")
    if not os.path.exists(test_path):
        continue

    for defect in os.listdir(test_path):
        defect_images = os.path.join(test_path, defect)
        if not os.path.exists(defect_images):
            continue

        for img_name in os.listdir(defect_images):
            img_path = os.path.join(defect_images, img_name)
            class_id = get_class(defect)

            if defect != "good":
                mask_path = os.path.join(gt_path, defect, img_name.replace(".png", "_mask.png"))
                if not os.path.exists(mask_path):
                    continue

                mask = cv2.imread(mask_path, 0)
                h, w = mask.shape
                coords = np.column_stack(np.where(mask > 0))
                if len(coords) == 0:
                    continue

                y_min, x_min = coords.min(axis=0)
                y_max, x_max = coords.max(axis=0)

                x_center = ((x_min + x_max) / 2) / w
                y_center = ((y_min + y_max) / 2) / h
                bbox_width = (x_max - x_min) / w
                bbox_height = (y_max - y_min) / h

                labels.append((class_id, x_center, y_center, bbox_width, bbox_height))
            else:
                labels.append((class_id,))  # good image → empty label

            image_paths.append(img_path)

print("Total images found:", len(image_paths))

# Train/val split
train_imgs, val_imgs, train_lbls, val_lbls = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42
)

# Save images and labels
def save_dataset(images, labels, img_dir, lbl_dir):
    for idx, (img_path, label) in enumerate(zip(images, labels)):
        category = img_path.split(os.sep)[-4]
        defect = img_path.split(os.sep)[-2]
        base_name = os.path.splitext(os.path.basename(img_path))[0]

        new_name = f"{category}_{defect}_{base_name}_{idx}"
        dest_img = os.path.join(img_dir, new_name + ".png")
        shutil.copy2(img_path, dest_img)

        label_file = os.path.join(lbl_dir, new_name + ".txt")
        if len(label) == 1:
            # good → empty label
            open(label_file, "w").close()
        else:
            class_id, xc, yc, w, h = label
            with open(label_file, "w") as f:
                f.write(f"{class_id} {xc} {yc} {w} {h}")

print("Saving training data...")
save_dataset(train_imgs, train_lbls, IMG_TRAIN, LBL_TRAIN)

print("Saving validation data...")
save_dataset(val_imgs, val_lbls, IMG_VAL, LBL_VAL)

print("Dataset conversion complete!")

Scanning MVTec dataset...
Total images found: 1725
Saving training data...
Saving validation data...
Dataset conversion complete!


In [38]:
import os

print(len(os.listdir("dataset_yolo/images/train")))
print(len(os.listdir("dataset_yolo/labels/train")))

1380
1380


<h1>Model Finetuning</h1>

In [33]:
import sys
!{sys.executable} -m pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.9/823.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 27.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics] [ultralytics]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [34]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/keerthi04/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [35]:
!yolo detect train data=dataset.yaml model=yolov8n.pt epochs=30

Ultralytics 8.4.21 🚀 Python-3.10.18 torch-2.8.0 CPU (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretraine

In [2]:
import cv2
from ultralytics import YOLO

model = YOLO("models/best.pt")

# results = model.predict("016.png", conf=0.25)
results = model.predict("dataset/mvtec_anomaly_detection/capsule/train/good/001.png", conf=0.25)

img = results[0].orig_img
boxes = results[0].boxes

if boxes is None or len(boxes) == 0:
    label = "GOOD"
else:
    classes = boxes.cls.cpu().numpy()
    label = "MAJOR DEFECT" if 1 in classes else "MINOR DEFECT"

print("Prediction:", label)
# cv2.putText(img, label, (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

# # cv2.imshow("Prediction", img)
# cv2.waitKey(0)


image 1/1 /Users/keerthi04/Mine/PersonalProjects/DefectDetector/dataset/mvtec_anomaly_detection/capsule/train/good/001.png: 640x640 (no detections), 38.1ms
Speed: 1.7ms preprocess, 38.1ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Prediction: GOOD


In [3]:
import cv2
from ultralytics import YOLO
from collections import Counter

# Load trained model
model = YOLO("models/best.pt")


def predict_image(image_path):
    results = model.predict(image_path, conf=0.25, verbose=False)

    boxes = results[0].boxes

    if boxes is None or len(boxes) == 0:
        return "GOOD"

    classes = boxes.cls.cpu().numpy()

    if 1 in classes:
        return "MAJOR DEFECT"
    else:
        return "MINOR DEFECT"


def majority_vote(image_list):

    predictions = []

    for img_path in image_list:
        pred = predict_image(img_path)
        predictions.append(pred)
        print(f"{img_path} -> {pred}")

    # Count predictions
    vote = Counter(predictions)

    final_result = vote.most_common(1)[0][0]

    print("\nVote Summary:", vote)
    print("Final Decision:", final_result)

    return final_result


# Example images of same product
images = [
    "dataset/mvtec_anomaly_detection/capsule/train/good/001.png",
    "dataset/mvtec_anomaly_detection/capsule/train/good/002.png",
    "dataset/mvtec_anomaly_detection/capsule/train/good/003.png",
    "dataset/mvtec_anomaly_detection/capsule/train/good/004.png",
    "dataset/mvtec_anomaly_detection/capsule/train/good/005.png"
]

majority_vote(images)

dataset/mvtec_anomaly_detection/capsule/train/good/001.png -> GOOD
dataset/mvtec_anomaly_detection/capsule/train/good/002.png -> GOOD
dataset/mvtec_anomaly_detection/capsule/train/good/003.png -> GOOD
dataset/mvtec_anomaly_detection/capsule/train/good/004.png -> GOOD
dataset/mvtec_anomaly_detection/capsule/train/good/005.png -> GOOD

Vote Summary: Counter({'GOOD': 5})
Final Decision: GOOD


'GOOD'